# Phase 5.2.1 — Offline Retrieval Replay
Run all cells in order. Reads the existing Phase 5.2 ZIP from Google Drive. No new DeepSeek requests, no embedding model, no package installation. Experimental policies are provisional, not independently validated.

In [ ]:
from pathlib import Path
import subprocess, importlib.util, sys
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-phase521')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
# Load only this standalone module; avoid stale imports and heavyweight dependencies.
path = ROOT/'src/tfl_agent/agent/phase52_replay.py'
spec = importlib.util.spec_from_file_location('phase52_replay',path)
replay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(replay)
print('Loaded:',path)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import zipfile, json, shutil
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
SOURCE_DIR = BASE/'Phase52_Outputs'
# Notebook 10 saves the original name (without browser download suffix).
ARCHIVE_NAME = '20261127T021948099654Z.zip'
SOURCE = SOURCE_DIR/ARCHIVE_NAME
if not SOURCE.is_file():
    alternative = SOURCE_DIR/'20261127T021948099654Z (1).zip'
    if alternative.is_file():
        SOURCE = alternative
    else:
        raise FileNotFoundError(f'Expected archive: {SOURCE}. Available: {[p.name for p in SOURCE_DIR.glob("*.zip")]}. Set ARCHIVE_NAME to the correct saved ZIP.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/Phase521')/RUN_ID
print('Source:',SOURCE)
print('Output:',OUT)

In [ ]:
report = replay.run(SOURCE, OUT, expected_n=19, expected_calls=44)
print('Cases:',report['cases'])
print('Historical completed Phase 3 calls:',report['historical_completed_phase3_calls'])
print('Baseline verified:',report['baseline_verified'])
print('New LLM calls:',report['new_llm_calls'])
print('New embedding calls:',report['new_embedding_calls'])
import pandas as pd
from IPython.display import display
for group in ('all','type_consistent','type_conflict'):
    print(group)
    display(pd.DataFrame(report['metrics'][group]).T)
print('Changed cases:')
changes = pd.read_csv(OUT/'Case_Changes.csv')
display(changes[changes.baseline_correct != changes.optimized_correct])

In [ ]:
DEST = BASE/'Phase521_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved = Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved to Google Drive:',saved)
# Optional: download only the output ZIP from the Drive UI if you want to share it.